# 04 — Gallery, compatibility và inference

Notebook này kiểm tra catalog/mapping/gallery hiện hành bằng đúng module runtime. Nó không tự build gallery, evaluate hay inference. Gallery hiện chỉ phủ năm item AXE; detection của tám brand khác phải tiếp tục unresolved thay vì bị force-map sang AXE.

## Khởi chạy

Chạy từ thư mục gốc của repository trong PowerShell:

```powershell
Set-Location D:/code/anhoa/train_embedding
./.venv/Scripts/Activate.ps1
jupyter lab notebooks/04_gallery_and_inference.ipynb
```

Notebook không có ô cài package. Không bật cờ build/evaluation/inference nếu chỉ muốn kiểm tra cấu hình và artifact metadata.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT_CANDIDATES = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repository root hoặc thư mục notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Các import này cũng được gallery CLI và backend recognition sử dụng.
from src.catalog.catalog import load_catalog, require_usable_catalog
from src.catalog.mapping import load_mapping
from src.catalog.profiles import load_product_profile
from src.config import load_brand_registry, resolve_repo_path
from src.recognition.pipeline import (
    MISSING_GALLERY_REASON,
    PanoramaRecognizer,
    RecognitionConfig,
    gallery_supported_brands,
)
from src.retrieval.item_code import GalleryIndex, validate_gallery_runtime_compatibility

CONFIG = {
    'backend_config': 'configs/panorama/backend.yaml',
    'checkpoint': 'checkpoints/axe_5sku/classification_100ep_s42/best_model.pth',
    'catalog': 'artifacts/catalog/axe5_demo_v3/product_catalog.json',
    'mapping': 'artifacts/mappings/axe5_classification_100ep_s42_axe5_demo_mapping.v3.json',
    'gallery_dir': 'artifacts/galleries/axe5_classification_100ep_s42_axe5_demo_v3',
    'dataset_dir': 'data/axe_crops',
    'inference_image': None,  # Gán path panorama/crop thật chỉ khi chủ động inference.
}
RUN_GALLERY_BUILD = False
RUN_VALIDATION_EVALUATION = False
RUN_TEST_EVALUATION = False
RUN_INFERENCE = False
display(Markdown(f'**Repository:** `{PROJECT_ROOT}`'))

## Registry và product identity

Class detector và item identity được đọc từ config canonical, không hardcode lại trong notebook.

In [ ]:
registry = load_brand_registry()
profile = load_product_profile('axe5')
display(pd.DataFrame([{'class_id': brand.id, 'brand': brand.name, 'aliases': ', '.join(brand.aliases)} for brand in registry.profile('brand_9brands')]))
display(pd.DataFrame([
    {
        'class_index': row.class_index,
        'legacy_class_name': row.legacy_class_name,
        'item_code': row.item_code,
    }
    for row in profile.classes
]))
display(pd.DataFrame([profile.identity_guard('blue_lavender')], index=['blue_lavender_guard']))

## Load và kiểm tra artifact chain

`GalleryIndex.load` kiểm tra shape, L2-normalization declaration, required metadata, version consistency và việc cosine không phải calibrated probability. Cell chỉ đọc gallery nhỏ hiện có; không load checkpoint/model.

In [ ]:
catalog_path = resolve_repo_path(CONFIG['catalog'])
mapping_path = resolve_repo_path(CONFIG['mapping'])
gallery_dir = resolve_repo_path(CONFIG['gallery_dir'])
catalog = load_catalog(catalog_path)
require_usable_catalog(catalog, require_scope=True)
mapping = load_mapping(mapping_path)
gallery = GalleryIndex.load(gallery_dir)

validate_gallery_runtime_compatibility(
    gallery.manifest,
    model_version=gallery.manifest['model_version'],
    catalog_version=catalog['catalog_version'],
    mapping_version=mapping['mapping_version'],
    preprocessing_version=gallery.manifest['preprocessing_version'],
    embedding_dimension=gallery.embeddings.shape[1],
)
coverage = gallery_supported_brands(catalog, gallery.metadata)
summary = {
    'catalog_status': catalog.get('status'),
    'catalog_version': catalog.get('catalog_version'),
    'mapping_status': mapping.get('status'),
    'mapping_version': mapping.get('mapping_version'),
    'gallery_entries': len(gallery.metadata),
    'gallery_items': gallery.manifest.get('item_count'),
    'embedding_dimension': gallery.embeddings.shape[1],
    'gallery_supported_brands': sorted(coverage.values()),
    'missing_gallery_reason': MISSING_GALLERY_REASON,
    'cosine_is_probability': gallery.manifest.get('cosine_is_calibrated_probability'),
}
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))

In [ ]:
item_counts = Counter(str(row['item_code']) for row in gallery.metadata)
gallery_table = pd.DataFrame([
    {
        'item_code': code,
        'gallery_images': count,
        'item_name': next(row['item_name'] for row in gallery.metadata if str(row['item_code']) == code),
    }
    for code, count in sorted(item_counts.items())
])
display(gallery_table)
ax = gallery_table.plot.bar(x='item_code', y='gallery_images', legend=False, figsize=(9, 4), rot=0)
ax.set_title('Gallery coverage theo item code')
ax.set_ylabel('Số ảnh')
plt.show()

## Gallery build — opt-in

Không ghi đè gallery v3. Nếu thật sự cần build, chọn output version mới và review catalog/mapping/checkpoint trước khi bật cờ.

In [ ]:
def run_checked(command: list[str]) -> None:
    print(subprocess.list2cmdline(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

gallery_output = 'artifacts/galleries/axe5_manual_review_v4'
gallery_command = [
    sys.executable, 'scripts/build_item_gallery.py',
    '--checkpoint', CONFIG['checkpoint'],
    '--catalog', CONFIG['catalog'],
    '--mapping', CONFIG['mapping'],
    '--dataset-dir', CONFIG['dataset_dir'],
    '--output-dir', gallery_output,
    '--device', 'auto',
]
if RUN_GALLERY_BUILD:
    if resolve_repo_path(gallery_output).exists():
        raise FileExistsError(f'Không ghi đè gallery: {resolve_repo_path(gallery_output)}')
    run_checked(gallery_command)
else:
    print('RUN_GALLERY_BUILD=False — gallery hiện hành không bị thay đổi.')

## Validation và test evaluation — hai cờ riêng

Validation command không truyền query manifest nên CLI dùng `val.csv`. Test bắt buộc truyền `test.csv` và không được dùng để chọn checkpoint hoặc threshold.

In [ ]:
base_evaluation_command = [
    sys.executable, 'scripts/evaluate_item_retrieval.py',
    '--checkpoint', CONFIG['checkpoint'],
    '--gallery-dir', CONFIG['gallery_dir'],
    '--mapping', CONFIG['mapping'],
    '--dataset-dir', CONFIG['dataset_dir'],
    '--device', 'auto',
]
validation_command = base_evaluation_command + [
    '--output-dir', 'artifacts/evaluations/manual_validation_review',
]
if RUN_VALIDATION_EVALUATION:
    run_checked(validation_command)
else:
    print('RUN_VALIDATION_EVALUATION=False')

test_manifest = resolve_repo_path('data/axe_crops/manifests/test.csv')
test_command = base_evaluation_command + [
    '--query-manifest', str(test_manifest),
    '--output-dir', 'artifacts/evaluations/manual_test_opt_in',
]
if RUN_TEST_EVALUATION:
    if not test_manifest.is_file():
        raise FileNotFoundError(test_manifest)
    run_checked(test_command)
else:
    print('RUN_TEST_EVALUATION=False — test split không được đọc.')

## Panorama inference — opt-in

Cell dùng trực tiếp `PanoramaRecognizer`, cùng pipeline với backend. Contract panorama là OpenCV BGR; ảnh Pillow RGB được đổi channel order đúng một lần trước khi gọi recognizer. Chỉ gán `CONFIG['inference_image']` và bật `RUN_INFERENCE` khi muốn chạy một ảnh cụ thể; model/checkpoint không được load ở trạng thái mặc định.

In [ ]:
if RUN_INFERENCE:
    if not CONFIG['inference_image']:
        raise ValueError('Hãy cấu hình inference_image trước khi bật RUN_INFERENCE.')
    import numpy as np
    from PIL import Image

    image_path = resolve_repo_path(CONFIG['inference_image'])
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    raw_config = RecognitionConfig.from_yaml(resolve_repo_path(CONFIG['backend_config']))
    runtime_config = replace(
        raw_config,
        yolo_weights=str(resolve_repo_path(raw_config.yolo_weights)),
        embedding_checkpoint=str(resolve_repo_path(raw_config.embedding_checkpoint)),
        catalog=str(resolve_repo_path(raw_config.catalog)),
        gallery_dir=str(resolve_repo_path(raw_config.gallery_dir)),
        mapping=str(resolve_repo_path(raw_config.mapping)),
    )
    recognizer = PanoramaRecognizer(runtime_config)
    # PanoramaRecognizer nhận BGR giống OpenCV/backend; Pillow trả RGB.
    panorama = np.asarray(Image.open(image_path).convert('RGB'))[:, :, ::-1].copy()
    inference_result = recognizer.recognize(panorama)
    display(pd.DataFrame(inference_result.get('detections', [])))
    display(inference_result.get('summary', inference_result))
else:
    print('RUN_INFERENCE=False — không load model và không chạy inference.')

## Lưu ý

- Không rebuild gallery chỉ để mở notebook.
- Catalog coverage, checkpoint class coverage và gallery coverage là ba khái niệm riêng.
- Brand không có gallery phải ở trạng thái `missing_gallery_for_detected_brand`.
- Cosine similarity không được hiển thị như phần trăm xác suất.
- `64823658` spray và `64863894` body wash là hai identity khác nhau.